# WIS2 pygeoAPI — Query Notebook

This notebook demonstrates how to query the IMOS WIS2 node OGC API (powered by **pygeoapi**).

**Base URL:** `https://wis2box.production.aodn.org.au/oapi`

**Collections available:**

| Collection | Contents |
|---|---|
| `discovery-metadata` | WCMP2 dataset records |
| `stations` | WIGOS-registered wave buoy stations |
| `messages` | WIS Notification Messages (with BUFR download links) |
| `urn:wmo:md:au-imos:wave-buoys` | Decoded observations (GeoJSON) |

Reference: [`docs/pygeoAPI.md`](../../docs/pygeoAPI.md)


## 0. Setup

In [1]:
import requests
import json
import pandas as pd

BASE_URL = "https://wis2box.production.aodn.org.au/oapi"

def get(path, **params):
    """GET helper — always requests JSON, prints status."""
    params.setdefault("f", "json")
    r = requests.get(f"{BASE_URL}{path}", params=params)
    r.raise_for_status()
    return r.json()


## 1. API Landing Page

The landing page lists all available links including the OpenAPI spec and collections endpoint.


In [2]:
landing = get("")
for link in landing["links"]:
    print(f"{link['rel']:20s}  {link['href']}")


about                 https://docs.wis2box.wis.wmo.int
self                  https://wis2box.production.aodn.org.au/oapi?f=json
alternate             https://wis2box.production.aodn.org.au/oapi?f=jsonld
alternate             https://wis2box.production.aodn.org.au/oapi?f=html
service-desc          https://wis2box.production.aodn.org.au/oapi/openapi
service-doc           https://wis2box.production.aodn.org.au/oapi/openapi?f=html
conformance           https://wis2box.production.aodn.org.au/oapi/conformance
data                  https://wis2box.production.aodn.org.au/oapi/collections
http://www.opengis.net/def/rel/ogc/1.0/processes  https://wis2box.production.aodn.org.au/oapi/processes
http://www.opengis.net/def/rel/ogc/1.0/job-list  https://wis2box.production.aodn.org.au/oapi/jobs
http://www.opengis.net/def/rel/ogc/1.0/tiling-schemes  https://wis2box.production.aodn.org.au/oapi/TileMatrixSets?f=json
http://www.opengis.net/def/rel/ogc/1.0/tiling-schemes  https://wis2box.production.aodn.org

## 2. List Collections

Each collection has an `id`, a human-readable `title`, and spatial/temporal extent information.


In [3]:
collections = get("/collections")
for c in collections["collections"]:
    print(f"{c['id']:45s}  {c['title']}")


discovery-metadata                             Discovery metadata
stations                                       Stations
urn:wmo:md:au-imos:wave-buoys                  Observations in json format for urn:wmo:md:au-imos:wave-buoys
messages                                       Data notifications


## 3. Discovery Metadata

The `discovery-metadata` collection contains WCMP2 records describing published datasets.


In [4]:
records = get("/collections/discovery-metadata/items")
print(f"Total records: {records['numberMatched']}")
for f in records["features"]:
    print(f"  {f['id']}")
    print(f"    title: {f['properties'].get('title', '')}")


Total records: 1
  urn:wmo:md:au-imos:wave-buoys
    title: Wave buoy observations made as part of the -- IMOS-WIS2.0


In [5]:
# Fetch the wave-buoys record directly
record = get("/collections/discovery-metadata/items/urn:wmo:md:au-imos:wave-buoys")
print(json.dumps(record["properties"], indent=2, default=str))


{
  "identifier": "urn:wmo:md:au-imos:wave-buoys",
  "title": "Wave buoy observations made as part of the -- IMOS-WIS2.0",
  "description": "Australian IMOS-WIS2.0 coastal wave buoy observations",
  "themes": [
    {
      "concepts": [
        {
          "id": "ocean"
        }
      ],
      "scheme": "https://codes.wmo.int/wis/topic-hierarchy/earth-system-discipline"
    }
  ],
  "type": "dataset",
  "created": "2025-07-30T00:00:00Z",
  "updated": "2026-02-13T03:54:25Z",
  "contacts": [
    {
      "addresses": [
        {
          "deliveryPoint": [
            "GPO Box 367, Hobart"
          ],
          "city": "Hobart",
          "administrativeArea": "Hobart",
          "postalCode": "7001",
          "country": "Australia"
        }
      ],
      "roles": [
        "host"
      ],
      "organization": "Integrated Marine Observing System (IMOS)",
      "name": "Integrated Marine Observing System (IMOS)",
      "position": "Integrated Marine Observing System (IMOS)",
      "

## 4. Stations

The `stations` collection lists all 23 WIGOS-registered wave buoy stations.

**Queryable fields:** `wigos_station_identifier`, `name`, `facility_type`,
`territory_name`, `wmo_region`, `status`, `topic`


In [6]:
# All stations as a DataFrame
result = get("/collections/stations/items", limit=100)
print(f"Total stations: {result['numberMatched']}")

stations_df = pd.json_normalize([f["properties"] for f in result["features"]])
stations_df = stations_df[["name", "wigos_station_identifier", "status",
                            "facility_type", "wmo_region", "territory_name"]]
stations_df


Total stations: 23


,name,wigos_station_identifier,status,facility_type,wmo_region,territory_name
0,TORBAY-WEST,0-22000-0-5601877,operational,seaFixed,southWestPacific,AUS
1,OCEAN-BEACH,0-22000-0-5601874,operational,seaFixed,southWestPacific,AUS
2,CORAL-BAY,0-22000-0-5601875,operational,seaFixed,southWestPacific,AUS
3,SHARK-BAY,0-22000-0-5601876,operational,seaFixed,southWestPacific,AUS
4,STORM-BAY,0-22000-0-5501882,operational,seaFixed,southWestPacific,AUS
5,HILLARYS,0-22000-0-5601873,operational,seaFixed,southWestPacific,AUS
6,BRIGHTON,0-22000-0-5501881,operational,seaFixed,southWestPacific,AUS
7,ROBE,0-22000-0-5501873,operational,seaFixed,southWestPacific,AUS
8,APOLLO-BAY,0-22000-0-7811080,operational,seaFixed,southWestPacific,AUS
9,NORTH-KANGAROO-ISLAND,0-22000-0-5501870,operational,seaFixed,southWestPacific,AUS


In [7]:
# Fetch a single station by WIGOS ID
wigos_id = "0-22000-0-7811080"
station = get(f"/collections/stations/items/{wigos_id}")
print(json.dumps(station["properties"], indent=2))


{
  "name": "APOLLO-BAY",
  "wigos_station_identifier": "0-22000-0-7811080",
  "traditional_station_identifier": "7811080",
  "barometer_height": 0.0,
  "facility_type": "seaFixed",
  "territory_name": "AUS",
  "wmo_region": "southWestPacific",
  "url": "https://oscar.wmo.int/surface/#/search/station/stationReportDetails/0-22000-0-7811080",
  "topic": "origin/a/wis2/au-imos/data/core/ocean/surface-based-observations/wave-buoys",
  "topics": [
    "origin/a/wis2/au-imos/data/core/ocean/surface-based-observations/wave-buoys"
  ],
  "status": "operational",
  "id": "0-22000-0-7811080"
}


In [8]:
# Stations within a bounding box (south-east Australia)
result = get("/collections/stations/items", bbox="140,-40,155,-30", limit=50)
print(f"Stations in SE Australia: {result['numberMatched']}")
for f in result["features"]:
    p = f["properties"]
    lon, lat = f["geometry"]["coordinates"][:2]
    print(f"  {p['name']:25s}  {wigos_id:25s}  ({lat:.4f}, {lon:.4f})")


Stations in SE Australia: 8
  APOLLO-BAY                 0-22000-0-7811080          (-38.7541, 143.7232)
  COLLAROY-NARRABEEN         0-22000-0-7811080          (-33.7264, 151.3068)
  CAPE-BRIDGEWATER           0-22000-0-7811080          (-38.3599, 141.2739)
  WILSONS-PROM               0-22000-0-7811080          (-39.5379, 146.4834)
  BENGELLO                   0-22000-0-7811080          (-35.8800, 150.1612)
  CENTRAL                    0-22000-0-7811080          (-38.0622, 144.8672)
  TATHRA                     0-22000-0-7811080          (-36.7083, 150.0010)
  BOB                        0-22000-0-7811080          (-38.6256, 142.3831)


## 5. Notifications (`messages`)

The `messages` collection contains one **WIS Notification Message (WNM)** per published BUFR file.
Each feature includes:
- `properties.data_id` — unique identifier
- `properties.datetime` — observation time
- `properties.pubtime` — publication time
- `properties.wigos_station_identifier`
- `links[].href` (rel=`canonical`) — direct BUFR4 download URL

**Queryable fields:** `wigos_station_identifier`, `data_id`, `datetime`, `pubtime`, `metadata_id`


In [9]:
# Check queryable fields
queryables = get("/collections/messages/queryables")
for name, info in queryables["properties"].items():
    print(f"  {name:35s}  {info.get('type', '')}")


  geometry                             
  data_id                              string
  datetime                             string
  id                                   string
  metadata_id                          string
  phenomenonTime                       string
  pubTime                              string
  pubtime                              string
  resultTime                           string
  value                                number
  wigos_station_identifier             string


In [10]:
# Latest 10 notifications for Apollo Bay
wigos_id = "0-22000-0-7811080"
result = get("/collections/messages/items",
             wigos_station_identifier=wigos_id, limit=10)

print(f"Total notifications for {wigos_id}: {result['numberMatched']}")

rows = []
for f in result["features"]:
    p = f["properties"]
    bufr_url = next((l["href"] for l in f["links"] if l["rel"] == "canonical"), None)
    rows.append({
        "datetime":   p["datetime"],
        "pubtime":    p["pubtime"],
        "data_id":    p["data_id"],
        "bufr_url":   bufr_url,
    })

pd.DataFrame(rows)


Total notifications for 0-22000-0-7811080: 2764


,datetime,pubtime,data_id,bufr_url
0,2025-11-11T10:20:00Z,2025-11-11T11:18:09Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
1,2025-11-11T11:20:00Z,2025-11-11T12:18:02Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
2,2025-11-11T12:20:00Z,2025-11-11T13:17:57Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
3,2025-11-11T07:20:00Z,2025-11-11T08:18:07Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
4,2025-11-11T08:20:00Z,2025-11-11T09:18:03Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
5,2025-11-11T13:20:00Z,2025-11-11T14:18:22Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
6,2025-11-11T09:20:00Z,2025-11-11T10:18:02Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
7,2025-11-11T14:20:00Z,2025-11-11T15:18:10Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
8,2025-11-12T00:20:00Z,2025-11-12T01:18:18Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...
9,2025-11-12T08:20:00Z,2025-11-12T09:18:27Z,au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000...,https://wis2box.production.aodn.org.au/data/20...


In [13]:
# Filter by station + datetime range
result = get("/collections/messages/items",
             wigos_station_identifier="0-22000-0-7811080",
             datetime="2026-05-06/2026-05-07",
             limit=10)

print(f"Matched: {result['numberMatched']}")
for f in result["features"]:
    p = f["properties"]
    print(f"  {p['datetime']}  {p['data_id']}")


Matched: 28
  2026-05-06T07:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T070000
  2026-05-06T04:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T040000
  2026-05-06T00:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T000000
  2026-05-06T01:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T010000
  2026-05-06T03:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T030000
  2026-05-06T05:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T050000
  2026-05-06T06:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T060000
  2026-05-06T08:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T080000
  2026-05-06T09:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T090000
  2026-05-06T13:00:00Z  au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260506T130000


In [36]:
# Spatial filter — all notifications within a bounding box
result = get("/collections/messages/items",
             bbox="143.0,-39.0,144.5,-38.0",
             limit=5)

print(f"Notifications in bbox: {result['numberMatched']}")
for f in result["features"]:
    p = f["properties"]
    print(f"  {p['datetime']}  {p['data_id'][:60]}")


Notifications in bbox: 2949
  2025-07-30T00:00:00Z  au-bom-imos/metadata/urn:wmo:md:au-bom-imos:wave-buoy-apollo
  2025-07-30T00:00:00Z  au-bom-imos/metadata/urn:wmo:md:au-bom-imos:wave-buoy-apollo
  2025-11-11T10:20:00Z  au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000-0-7811080_202
  2025-11-11T11:20:00Z  au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000-0-7811080_202
  2025-11-11T12:20:00Z  au-bom-imos:wave-buoy-apollo-bay/WIGOS_0-22000-0-7811080_202


In [32]:
result

{'type': 'FeatureCollection',
 'features': [{'id': '49a2bdbe-59b0-46ec-afb8-72622894eeb8',
   'type': 'Feature',
   'conformsTo': ['http://wis.wmo.int/spec/wnm/1/conf/core'],
   'geometry': {'type': 'Polygon',
    'coordinates': [[[143.7223, -38.75452],
      [143.7223, -38.7535],
      [143.72362, -38.7535],
      [143.72362, -38.75452],
      [143.7223, -38.75452]]]},
   'properties': {'data_id': 'au-bom-imos/metadata/urn:wmo:md:au-bom-imos:wave-buoy-apollo-bay',
    'datetime': '2025-07-30T00:00:00Z',
    'pubtime': '2025-11-10T23:29:26Z',
    'integrity': {'method': 'sha512',
     'value': 'eM0/Dzz808hOOpvMP3w8mWE4thRGlw+H8OnnJkNl6Z9ml/v4guPkq1em6s6sldC2Kv83Xb5zjN5BhWIrmFiLCg=='},
    'content': {'encoding': 'base64',
     'value': 'eyJpZCI6ICJ1cm46d21vOm1kOmF1LWJvbS1pbW9zOndhdmUtYnVveS1hcG9sbG8tYmF5IiwgImNvbmZvcm1zVG8iOiBbImh0dHA6Ly93aXMud21vLmludC9zcGVjL3djbXAvMi9jb25mL2NvcmUiXSwgInR5cGUiOiAiRmVhdHVyZSIsICJnZW9tZXRyeSI6IHsidHlwZSI6ICJQb2x5Z29uIiwgImNvb3JkaW5hdGVzIjogW1tbMTQzLjcyM

## 6. Pagination

The API is offset-based. Use `numberMatched` to know the total and increment `offset` by `limit`.


In [16]:
def fetch_all_messages(wigos_id, datetime_range, page_size=100):
    """Fetch all notifications for a station in a date range."""
    params = {
        "wigos_station_identifier": wigos_id,
        "datetime": datetime_range,
        "limit": page_size,
        "offset": 0,
    }
    features = []
    while True:
        page = get("/collections/messages/items", **params)
        features.extend(page["features"])
        total = page["numberMatched"]
        print(f"  Fetched {len(features)}/{total}")
        if len(features) >= total:
            break
        params["offset"] += page_size
    return features

features = fetch_all_messages("0-22000-0-7811080", "2025-11-01/2025-11-30")
print(f"\nTotal fetched: {len(features)}")


  Fetched 100/467
  Fetched 200/467
  Fetched 300/467
  Fetched 400/467
  Fetched 467/467

Total fetched: 467


## 7. Download BUFR Files

Each notification's `links` array contains a `canonical` link to the BUFR4 file.


In [25]:
result = get("/collections/messages/items",
             wigos_station_identifier="0-22000-0-7811080",
             sortby="-datetime",
             limit=1)
result

{'type': 'FeatureCollection',
 'features': [{'id': '20e3866d-cf6b-4b5d-8af2-c54a14aee78a',
   'type': 'Feature',
   'conformsTo': ['http://wis.wmo.int/spec/wnm/1/conf/core'],
   'geometry': {'coordinates': [143.72267, -38.75338], 'type': 'Point'},
   'properties': {'data_id': 'au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260507T040000',
    'datetime': '2026-05-07T04:00:00Z',
    'pubtime': '2026-05-07T04:33:18Z',
    'integrity': {'method': 'sha512',
     'value': 'koMjtV/aDzx6P2HJ7kLSo8dcnHiuxCiulcOd5GslIumEH7btkaNCEa1KJR8tOBrRnDELda0vSSWMTujbLcB3iQ=='},
    'metadata_id': 'urn:wmo:md:au-imos:wave-buoys',
    'content': {'encoding': 'base64',
     'value': 'QlVGUgAAegQAABYAAAH//wAAAAZuKQAH6gUHBAAAAAAJAAABgMgPAABPACIrSP/////////////////v1KOQAnGRs9vsV//6sE3/////+EKQH/8l////////gP//////wH//65///+A//////8VAP///////4Dc3Nzc=',
     'size': 122},
    'wigos_station_identifier': '0-22000-0-7811080',
    'id': '20e3866d-cf6b-4b5d-8af2-c54a14aee78a'},
   'links': [{'rel': 'canonical',
     'ty

In [ ]:
# MQTT message content
feature = result["features"][0]
feature

{'id': '20e3866d-cf6b-4b5d-8af2-c54a14aee78a',
 'type': 'Feature',
 'conformsTo': ['http://wis.wmo.int/spec/wnm/1/conf/core'],
 'geometry': {'coordinates': [143.72267, -38.75338], 'type': 'Point'},
 'properties': {'data_id': 'au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260507T040000',
  'datetime': '2026-05-07T04:00:00Z',
  'pubtime': '2026-05-07T04:33:18Z',
  'integrity': {'method': 'sha512',
   'value': 'koMjtV/aDzx6P2HJ7kLSo8dcnHiuxCiulcOd5GslIumEH7btkaNCEa1KJR8tOBrRnDELda0vSSWMTujbLcB3iQ=='},
  'metadata_id': 'urn:wmo:md:au-imos:wave-buoys',
  'content': {'encoding': 'base64',
   'value': 'QlVGUgAAegQAABYAAAH//wAAAAZuKQAH6gUHBAAAAAAJAAABgMgPAABPACIrSP/////////////////v1KOQAnGRs9vsV//6sE3/////+EKQH/8l////////gP//////wH//65///+A//////8VAP///////4Dc3Nzc=',
   'size': 122},
  'wigos_station_identifier': '0-22000-0-7811080',
  'id': '20e3866d-cf6b-4b5d-8af2-c54a14aee78a'},
 'links': [{'rel': 'canonical',
   'type': 'application/bufr',
   'href': 'https://wis2box.production.aodn.org.au/

In [26]:

feature = result["features"][0]
props   = feature["properties"]
bufr_url = next(l["href"] for l in feature["links"] if l["rel"] == "canonical")

print(f"data_id  : {props['data_id']}")
print(f"datetime : {props['datetime']}")
print(f"BUFR URL : {bufr_url}")

bufr_bytes = requests.get(bufr_url).content
print(f"\nDownloaded {len(bufr_bytes)} bytes")
print(f"BUFR magic bytes: {bufr_bytes[:4]}")   # should be b'BUFR'


data_id  : au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260507T040000
datetime : 2026-05-07T04:00:00Z
BUFR URL : https://wis2box.production.aodn.org.au/data/2026-05-07/wis/urn:wmo:md:au-imos:wave-buoys/WIGOS_0-22000-0-7811080_20260507T040000.bufr4

Downloaded 122 bytes
BUFR magic bytes: b'BUFR'


In [19]:
# Save to disk
import pathlib

outdir = pathlib.Path("data/")
outdir.mkdir(parents=True, exist_ok=True)

filename = outdir / bufr_url.split("/")[-1]
filename.write_bytes(bufr_bytes)
print(f"Saved: {filename}")


Saved: data/WIGOS_0-22000-0-7811080_20251111T102000.bufr4


## Summary

| Task | Pattern |
|---|---|
| List collections | `GET /oapi/collections` |
| Get discovery record | `GET /oapi/collections/discovery-metadata/items/{id}` |
| All stations | `GET /oapi/collections/stations/items` |
| One station | `GET /oapi/collections/stations/items/{wigos_id}` |
| Notifications for station | `?wigos_station_identifier=0-22000-0-7811080` |
| Filter by date | `?datetime=2026-05-01/2026-05-07` |
| Spatial filter | `?bbox=minLon,minLat,maxLon,maxLat` |
| Free-text search | `?q=WIGOS_0-22000-0-7811080` |
| Paginate | `?limit=100&offset=N` |
| Download BUFR | Follow `links[rel=canonical].href` |

See [`docs/pygeoAPI.md`](../../docs/pygeoAPI.md) for the full reference.
